<h1>Phase 5: The Autonomous Financial Director</h1>
<h3>Dynamic Multi-Agent Orchestration & Real-Time Research Planning</h3>
<p><i>Building the next generation of institutional-grade AI research departments with Vinagent</i></p>

---

Welcome to the capstone of the **Multi-Agent in Finance** series. 

In Phase 1–4, we mastered **Static State Graphs**. We hardcoded every logic path, branch, and loop. But real-world institutional research is messy. A PM doesn't always know which tools will be needed until the query is asked.

Phase 5 introduces **Dynamic Task Orchestration**. We evolve from building "scripts" to building a **Research Director**—an LLM-powered Planner that can:
1. **Analyze the Request**: Determine if it's a sector comparison, a macro risk check, or a fundamental deep-dive.
2. **Hire Specialists**: Dynamically activate agents with specifically registered tools.
3. **Optimize Execution**: Run independent tasks in **parallel** and manage sequential data-flow automatically.

### The Dynamic Pipeline

```mermaid
graph TD
    UserQuery["User Query"] --> Planner{"Planner Agent\n(Dynamic TaskGraph)"}
    
    subgraph "Dynamic Execution (Parallel where possible)"
        Planner -->|Hires| Crawler["Market Data Crawler"]
        Planner -->|Hires| Searcher["Global News Searcher"]
        Planner -->|Hires| Quant["Quantitative Analyst"]
        Planner -->|Hires| Risk["Risk Assessor"]
    end
    
    Crawler --> Quant
    Searcher --> Risk
    
    Quant --> Aggregator["Aggregator Agent\n(Portfolio Strategist)"]
    Risk --> Aggregator
    
    Aggregator --> FinalReport["Final Strategic Report"]
```

## 1. Environment & Tools Initialization

We import our specialized Vietnamese market toolkit and the `primary_function` wrapper from `vinagent.register`.

In [1]:
# %pip install -U vinagent pandas matplotlib plotly vnstock yfinance scipy -q

import os
import sys
import asyncio
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from vinagent.agent.agent import Agent
from vinagent.multi_agent.dynamic_crew import DynamicCrew
from vinagent.register import primary_function

# Ensure local tools are findable
sys.path.append(os.getcwd())
from vnstock_tools import (
    fetch_stock_data_vn, 
    visualize_stock_data_vn, 
    plot_returns_vn, 
    search_api,
    calculate_black_litterman,
    optimize_portfolio,
    calculate_equilibrium_returns,
    get_current_time,
    calculate_stock_statistics,
    fetch_fundamental_ratios_vn,
)

# Ensure .env contains OPENAI_API_KEY and TAVILY_API_KEY
load_dotenv(find_dotenv('.env'))

llm = ChatOpenAI(model="gpt-4o-mini")
print("✅ Environment initialized.")

📦 **Vnstock 3.5.0 is available**
Current version: 3.4.2
Update: `pip install vnstock --upgrade`
Release history: https://vnstocks.com/docs/tai-lieu/lich-su-phien-ban

📦 **Vnai 2.4.0 is available**
Current version: 2.3.9
Update: `pip install vnai --upgrade`
Release history: https://pypi.org/project/vnai/#history

✅ Environment initialized.


## 2. Defining the Specialist Agent Pool & Granular Tool Registration

Each agent below acts as a "Department" in our firm. Since the current version of the library expects tool paths as strings in the constructor, we will initialize the agents with empty tools and then use the `tools_manager` to register our function objects explicitly.

In [3]:
# 1. Initialize Specialists with empty tools list to avoid Path conversion errors
agents = {
    "Market Data Crawler": Agent(
        description="""You fetch technical and fundamental stock data for the Vietnam market.
        You use tools to find historical prices, P/E ratios, dividend yields, and real-time market stats.""",
        llm=llm,
        tools=[]
    ),
    "News Searcher": Agent(
        description="""You gather real-time financial news, regulatory updates, and macroeconomic events.
        You identify catalysts or risks affecting specific sectors or tickers.""",
        llm=llm,
        tools=[]
    ),
    "Quantitative Analyst": Agent(
        description="""You analyze market data to derive statistical insights.
        IMPORTANT: Always use symbols (e.g. 'FPT', 'VCB') when calling tools. 
        If you don't have a physical DataFrame yet, just pass a placeholder string (like 'df_of_FPT') and the symbol; 
        the tool will automatically fetch the necessary historical data for you.
        Always use symbols (e.g. 'FPT', 'VCB') when calling tools. If you need a physical DataFrame for analysis (e.g. for statistics), you must provide the ticker symbol to the tool. If you don't have the data yet, the tool will auto-fetch it.""",
        llm=llm,
        tools=[]
    ),
    "Risk Assessment Agent": Agent(
        description="""You evaluate qualitative and quantitative risks for portfolio selection.
        Synthesize news and volatility data into a risk rating (Low, Medium, High).
        You can use search_api to find recent risk catalysts.""",
        llm=llm,
        tools=[]
    ),
    "Executive Strategist": Agent(
        description="""You are the final decision maker. 
        Synthesize all specialist reports into a final 'Institutional Advice' document..
        Ensure you highlight P/E ratios and Dividend Yields clearly.""",
        llm=llm,
        tools=[]
    )
}

# 2. Explicitly register function tools using tools_manager
# Market Data Crawler
for t in [fetch_stock_data_vn, get_current_time, fetch_fundamental_ratios_vn]:
    agents["Market Data Crawler"].tools_manager.register_function_tool(primary_function(t))

# News Searcher
agents["News Searcher"].tools_manager.register_function_tool(primary_function(search_api))

# Quantitative Analyst
for t in [visualize_stock_data_vn, plot_returns_vn, calculate_stock_statistics]:
    agents["Quantitative Analyst"].tools_manager.register_function_tool(primary_function(t))

# Risk Assessment Agent
agents["Risk Assessment Agent"].tools_manager.register_function_tool(primary_function(search_api))

# Executive Strategist
for t in [optimize_portfolio, calculate_black_litterman, calculate_equilibrium_returns]:
    agents["Executive Strategist"].tools_manager.register_function_tool(primary_function(t))

print(f"🏛️ Specialist pool initialized and tools registered for each department.")

INFO:vinagent.register.tool:Registered tool: fetch_stock_data_vn (runtime)
INFO:vinagent.register.tool:Registered tool: get_current_time (runtime)
INFO:vinagent.register.tool:Registered tool: fetch_fundamental_ratios_vn (runtime)
INFO:vinagent.register.tool:Registered tool: search_api (runtime)
INFO:vinagent.register.tool:Registered tool: visualize_stock_data_vn (runtime)
INFO:vinagent.register.tool:Registered tool: plot_returns_vn (runtime)
INFO:vinagent.register.tool:Registered tool: calculate_stock_statistics (runtime)
INFO:vinagent.register.tool:Registered tool: search_api (runtime)
INFO:vinagent.register.tool:Registered tool: optimize_portfolio (runtime)
INFO:vinagent.register.tool:Registered tool: calculate_black_litterman (runtime)
INFO:vinagent.register.tool:Registered tool: calculate_equilibrium_returns (runtime)


🏛️ Specialist pool initialized and tools registered for each department.


## 3. Launching the Research Director (`DynamicCrew`)

The `DynamicCrew` uses the LLM to generate a `TaskGraph` at runtime. The **Planner** will evaluate the query and 'hire' the specialists who now have their specific tools registered.

In [4]:
director = DynamicCrew(
    llm=llm,
    agents=agents
)

print("🚀 Dynamic Research Director ready for orchestration.")

🚀 Dynamic Research Director ready for orchestration.


## 4. Case Study: Cross-Sector Dividend & Risk Analysis

**The Query:**
> "Compare the Banking (VCB) and Technology (FPT) sectors in Vietnam. 
> 1. Fetch their fundamental highlights (Dividend Yield, P/E). 
> 2. Calculate their historical statistics. 
> 3. Search for recent catalysts in each sector.
> 4. Return a final strategic recommendation."


In [6]:
query = (
    "Compare the Banking (VCB) and Technology (FPT) sectors in Vietnam in 2026. "
    "Fetch their dividend yields and P/E ratios, calculate 1-year volatility/returns, "
    "and check for any regulatory news or catalysts for both sectors. "
    "Provide a final strategic risk-reward recommendation."
)

async def execute_research():
    print("--- SUBMITTING INSTITUTIONAL QUERY ---")
    # director.ainvoke() orchestrates the entire TaskGraph dynamically.
    result = await director.ainvoke(query)
    
    print("\n--- FINAL STRATEGIC REPORT ---")
    print(result)

# Jupyter supports top-level await
await execute_research()

INFO:vinagent.multi_agent.dynamic_crew:DynamicCrew: invoking Planner (async) …


--- SUBMITTING INSTITUTIONAL QUERY ---


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:vinagent.multi_agent.dynamic_crew:DynamicCrew: plan produced 5 step(s).
INFO:vinagent.multi_agent.dynamic_crew:DynamicCrew: plan produced 5 step(s).
INFO:vinagent.multi_agent.dynamic_crew:
Step 0: Retrieve dividend yield data for the Banking (VCB) and Technology (FPT) sectors in Vietnam for 2026.
  └─ Task ID: fetch_dividend_yields
  └─ Agent: Market Data Crawler
  └─ Depends On: []

Step 1: Retrieve P/E ratio data for the Banking (VCB) and Technology (FPT) sectors in Vietnam for 2026.
  └─ Task ID: fetch_pe_ratios
  └─ Agent: Market Data Crawler
  └─ Depends On: []

Step 2: Calculate the 1-year volatility and returns for the Banking (VCB) and Technology (FPT) sectors in Vietnam.
  └─ Task ID: calculate_volatility_returns
  └─ Agent: Quantitative Analyst
  └─ Depends On: ['fetch_dividend_yields', 'fetch_pe_ratios']

Step 3: Search for regulatory news or catalysts affecting the Banking (VCB) 

⚠️ VCI source missing PE for VCB, trying fallback SSI source...


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:vinagent.agent.agent:Response: requires_tool=False answer='I encountered errors fetching the P/E ratios for both VCB (Banking) and FPT (Technology) sectors in Vietnam. The results indicate that the data may not be available through the supported sources. Unfortunately, I cannot provide the requested P/E ratios for 2026.' tool_call=None fix_bug_command=None
INFO:vinagent.agent.agent:No more tool calls needed. Completed in 2 iterations.
INFO:vinagent.multi_agent.dynamic_crew:DynamicCrew.ainvoke: executing level 2/3 with 1 parallel step(s): ['calculate_volatility_returns']
INFO:vinagent.multi_agent.dynamic_crew:
Step 4: Calculate the 1-year volatility and returns for the Banking (VCB) and Technology (FPT) sectors in Vietnam.
  └─ Task ID: calculate_volatility_returns
  └─ Agent: Quantitative Analyst
  └─ Depends On: ['fetch_dividend_yields', 'fetch_pe_ratios']

INFO:vinagent.multi_agent.dynamic

🔄 Auto-fetching data for VCB (reason: tool received placeholder string)...


INFO:vinagent.register.tool:Completed executing function tool calculate_stock_statistics({'df': 'df_of_VCB', 'symbol': 'VCB'})
INFO:vinagent.agent.agent:Async tool calling iteration 2/5
INFO:vinagent.prompt.agent_prompt:Available Tools: {'fetch_stock_data_vn': {'tool_name': 'fetch_stock_data_vn', 'arguments': {'symbol': "<class 'str'>", 'start_date': "<class 'str'>", 'end_date': "<class 'str'>", 'interval': "<class 'str'>"}, 'return': "<class 'pandas.core.frame.DataFrame'>", 'docstring': "Fetch historical stock data from Vnstock.\n\n    Args:\n        symbol (str): The stock symbol (e.g., 'FPT' for FPT Corporation).\n        start_date (str): Start date for historical data (YYYY-MM-DD).\n        end_date (str): End date for historical data (YYYY-MM-DD).\n        interval (str): Data interval ('1d', '1wk', '1mo', etc.).\n\n    Returns:\n        pd.DataFrame: DataFrame containing historical stock prices.", 'module_path': '__runtime__', 'tool_type': 'function', 'tool_call_id': 'tool_a0422


--- FINAL STRATEGIC REPORT ---
### Comparative Analysis of the Banking (VCB) and Technology (FPT) Sectors in Vietnam (2026)

This report analyzes and compares the Banking sector, represented by Vietcombank (VCB), and the Technology sector, represented by FPT Corporation (FPT), in Vietnam for the year 2026. While certain key financial metrics such as dividend yields and P/E ratios could not be retrieved, insights regarding regulatory environments, volatility, returns, and strategic recommendations can guide investment decisions.

#### 1. Sector Overview and Regulatory Landscape

**Banking Sector (VCB)**:
- **Regulatory Environment**: The banking sector is experiencing a shift towards tighter regulatory standards. This is designed to support a new growth cycle, enhancing stability and transparency in the sector. However, the transition may bring temporary volatility as institutions adjust to new standards.
- **Opportunities**: Despite potential disruptions, this change is expected to fo

In [7]:
query = (
    "Compare the Banking (VCB) and natural resource (GAS) sectors in Vietnam. "
    "Fetch their dividend yields and P/E ratios, calculate 1-year volatility/returns in 2026,"
    "and check for any regulatory news or catalysts for both sectors. "
    "Provide a final strategic risk-reward recommendation."
)

async def execute_research():
    print("--- SUBMITTING INSTITUTIONAL QUERY ---")
    # director.ainvoke() orchestrates the entire TaskGraph dynamically.
    result = await director.ainvoke(query)
    
    print("\n--- FINAL STRATEGIC REPORT ---")
    print(result)

# Jupyter supports top-level await
await execute_research()

INFO:vinagent.multi_agent.dynamic_crew:DynamicCrew: invoking Planner (async) …


--- SUBMITTING INSTITUTIONAL QUERY ---


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:vinagent.multi_agent.dynamic_crew:DynamicCrew: plan produced 4 step(s).
INFO:vinagent.multi_agent.dynamic_crew:DynamicCrew: plan produced 4 step(s).
INFO:vinagent.multi_agent.dynamic_crew:
Step 0: Fetch dividend yields and P/E ratios of the Banking and natural resource sectors in Vietnam.
  └─ Task ID: fetch_dividend_yields_PE_ratios
  └─ Agent: Market Data Crawler
  └─ Depends On: []

Step 1: Calculate the 1-year volatility and returns for both sectors in 2026.
  └─ Task ID: calculate_volatility_returns
  └─ Agent: Quantitative Analyst
  └─ Depends On: []

Step 2: Check for any regulatory news or catalysts affecting both sectors.
  └─ Task ID: check_regulatory_news
  └─ Agent: News Searcher
  └─ Depends On: []

Step 3: Provide a final strategic risk-reward recommendation for both sectors based on gathered data.
  └─ Task ID: strategic_recommendation
  └─ Agent: Executive Strategist
  └─ Dep

⚠️ VCI source missing PE for GAS, trying fallback SSI source...


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:vinagent.logger.logger:tool_data: {
  "tool_name": "search_api",
  "tool_type": "function",
  "module_path": "__runtime__",
  "is_runtime": true,
  "arguments": {
    "query": "recent regulatory news and catalysts affecting the Banking sector (VCB) and Natural Resources sector (GAS) in Vietnam"
  },
  "return": "typing.Any"
}
INFO:vinagent.agent.agent:Response: requires_tool=True answer=None tool_call=ToolCall(tool_name='search_api', arguments={'query': 'recent regulatory news and catalysts affecting the Banking sector (VCB) and Natural Resources sector (GAS) in Vietnam'}, return_='typing.Any', module_path='__runtime__', tool_type='function', tool_call_id='tool_d324b401-ae5c-48c7-9cc8-f231e13f602', is_runtime=True) fix_bug_command=None
INFO:vinagent.logger.logger:AIMessage: content='' additional_kwargs={} response_metadata={} tool_calls=[{'name': 'search_api', 'args': {'query': 'recent regul


--- FINAL STRATEGIC REPORT ---
### Comparison of the Banking (VCB) and Natural Resource (GAS) Sectors in Vietnam

#### 1. Overview of Sectors

**Banking Sector (VCB):**
- The Vietnamese banking sector is currently experiencing a period of moderate growth, bolstered by a credit growth target set at 15%. This optimistic outlook reflects the anticipated economic recovery and increasing demand for banking services.

**Natural Resource Sector (GAS):**
- The GAS sector, which includes gas and LNG power generation, is facing transformative regulatory changes. Starting in March 2026, new carbon trading regulations are expected to impact operational strategies and investments, increasing uncertainty in the sector.

#### 2. Dividend Yields and P/E Ratios

Unfortunately, precise data on the current dividend yields and P/E ratios for both VCB and GAS could not be retrieved due to an error in the querying process. Therefore, it is advisable to consult financial databases or market reports directly

## 5. Summary & Progression

In this Phase 5 tutorial, we've demonstrated:

1. **Architectural Flexibility**: The system adapts to the query using high-level dynamic planners.
2. **Granular Tool Control**: We manually registered relevant tools to each agent after initialization to bypass constructor limitations.
3. **Dynamic Parameterization**: Agents use general-purpose tools with specific parameters, eliminating hardcoded function sprawl.

**You have now completed the Multi-Agent in Finance course!** 

From sequential scripts to dynamic research directors, you are now equipped to build the future of autonomous financial systems with **Vinagent**.